# Two Gaussian Beams Interfering

Create two tilted Gaussian beams, propagate them to the same detector plane, and show the interference fringes produced by summing their complex fields.


In [ ]:
import os
os.environ["JAX_ENABLE_X64"] = "1"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.2"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp

from temgym_core.components import Detector
from temgym_core.evaluate import evaluate_gaussians_for
from temgym_core.gaussian import make_gaussian
from temgym_core.run import run_to_end

jax.config.update("jax_enable_x64", True)


## Beam Pair

The beams share the same waist and voltage, but carry opposite angular tilts. Their phase difference creates fringes where the envelopes overlap.


In [ ]:
voltage = 200e3
waist = 25e-9
z_detector = 80e-6
window = 350e-9
shape = (192, 192)
pixel = window / shape[0]
detector = Detector(z=z_detector, pixel_size=(pixel, pixel), shape=shape)

beams = make_gaussian(
    x=jnp.array([-35e-9, 35e-9]),
    y=jnp.array([0.0, 0.0]),
    dx=jnp.array([1.2e-3, -1.2e-3]),
    dy=jnp.array([0.0, 0.0]),
    voltage=voltage,
    waist_x=waist,
    waist_y=waist,
    amp=jnp.array([1.0, 1.0]),
)
propagated = run_to_end(beams, (detector,))
field = np.asarray(evaluate_gaussians_for(propagated, detector))
individual_fields = [np.asarray(evaluate_gaussians_for(run_to_end(beams[i], (detector,)), detector)) for i in range(2)]
np.testing.assert_allclose(field, individual_fields[0] + individual_fields[1], rtol=1e-12, atol=1e-12)


## Fringes

The intensity is the measurable signal. The phase plot is included to show the two tilted phase ramps that produce the fringe spacing.


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)

im0 = ax[0].imshow(np.abs(individual_fields[0]), extent=detector.extent, origin="lower", cmap="inferno")
ax[0].set_title("Beam 1 amplitude")
fig.colorbar(im0, ax=ax[0])

im1 = ax[1].imshow(np.abs(field) ** 2, extent=detector.extent, origin="lower", cmap="inferno")
ax[1].set_title("Interference intensity")
fig.colorbar(im1, ax=ax[1])

center = shape[0] // 2
x_nm = np.asarray(detector.coords_1d[0]) * 1e9
ax[2].plot(x_nm, np.abs(field[center, :]) ** 2)
ax[2].set_title("Central intensity cut")
ax[2].set_xlabel("x (nm)")
ax[2].set_ylabel("intensity")
